In [2]:
import pickle
from pathlib import Path

import pandas as pd
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

C:\Users\PC-PIERRE\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
data = pd.read_csv('../data/dataset_TP_Flask.csv')

X = data.drop(columns=['Outcome'])
y = data['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")
data.head()

Train size: (614, 7), Test size: (154, 7)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,Age,Outcome
0,6,148,72,35,0,33.6,50,1
1,1,85,66,29,0,26.6,31,0
2,8,183,64,0,0,23.3,32,1
3,1,89,66,23,94,28.1,21,0
4,0,137,40,35,168,43.1,33,1


In [8]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced'
 )
rf_model.fit(X_train, y_train)

# Evaluation
y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_proba):.4f}")
print(classification_report(y_test, y_pred))

# SHAP explainer for tree model
explainer = shap.TreeExplainer(rf_model)

artifacts_dir = Path(r"C://Users//PC-PIERRE//Desktop//Miage//MASTER 2//Développement WEB//TD//TD1//explainer")
artifacts_dir.mkdir(parents=True, exist_ok=True)

with open(artifacts_dir / 'rf_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

with open(artifacts_dir / 'shap_explainer.pkl', 'wb') as f:
    pickle.dump(explainer, f)

with open(artifacts_dir / 'feature_names.pkl', 'wb') as f:
    pickle.dump(list(X.columns), f)

print(f"Saved: {artifacts_dir / 'rf_model.pkl'}")
print(f"Saved: {artifacts_dir / 'shap_explainer.pkl'}")
print(f"Saved: {artifacts_dir / 'feature_names.pkl'}")

Accuracy: 0.7403
ROC AUC: 0.8141
              precision    recall  f1-score   support

           0       0.80      0.80      0.80       100
           1       0.63      0.63      0.63        54

    accuracy                           0.74       154
   macro avg       0.71      0.71      0.71       154
weighted avg       0.74      0.74      0.74       154

Saved: C:\Users\PC-PIERRE\Desktop\Miage\MASTER 2\Développement WEB\TD\TD1\explainer\rf_model.pkl
Saved: C:\Users\PC-PIERRE\Desktop\Miage\MASTER 2\Développement WEB\TD\TD1\explainer\shap_explainer.pkl
Saved: C:\Users\PC-PIERRE\Desktop\Miage\MASTER 2\Développement WEB\TD\TD1\explainer\feature_names.pkl
